# 07 - ゲームテンポ分析

試合時間帯別勝率、ゴールドリード推移、逆転パターン、スノーボール率を分析する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df_matches = pd.read_csv(DATA / 'matches.csv', parse_dates=['gameCreation'])
df_frames = pd.read_csv(DATA / 'timeline_frames.csv')
df_players = pd.read_csv(DATA / 'player_stats.csv')

df_players['riotId'] = df_players['summonerName'] + '#' + df_players['tagLine']
member_teams = df_players[df_players['riotId'].isin(MEMBERS)][['matchId', 'teamId']].drop_duplicates()

In [ ]:
# 試合時間帯別勝率（短期戦 vs 長期戦）
match_with_team = df_matches.merge(member_teams, on='matchId', how='inner')
match_with_team['win'] = (
    (match_with_team['teamId'] == 100) & match_with_team['team100Win']
) | (
    (match_with_team['teamId'] == 200) & match_with_team['team200Win']
)

dur_bins = [0, 20, 25, 30, 35, 40, 45, 60]
dur_labels = ['<20分', '20-25', '25-30', '30-35', '35-40', '40-45', '45分+']
match_with_team['durBin'] = pd.cut(
    match_with_team['gameDurationMin'], bins=dur_bins, labels=dur_labels
)

dur_wr = match_with_team.groupby('durBin', observed=False).agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
).reset_index()
dur_wr['winrate_pct'] = (dur_wr['winrate'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

ax1.bar(dur_wr['durBin'].astype(str), dur_wr['games'], alpha=0.3, label='試合数')
ax2.plot(dur_wr['durBin'].astype(str), dur_wr['winrate_pct'], 'o-', color='red', label='勝率')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

ax1.set_xlabel('試合時間')
ax1.set_ylabel('試合数')
ax2.set_ylabel('勝率 %')
ax1.set_title('試合時間帯別 勝率 — 短期戦 vs 長期戦')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# ゴールドリード推移グラフ（分単位でチームゴールド差の推移）
team_gold_per_min = df_frames.groupby(
    ['matchId', 'timestampMin', 'teamId']
)['totalGold'].sum().reset_index()

pivoted = team_gold_per_min.pivot_table(
    index=['matchId', 'timestampMin'], columns='teamId', values='totalGold'
).reset_index()
pivoted.columns = ['matchId', 'timestampMin', 'gold_100', 'gold_200']
pivoted['goldDiff'] = pivoted['gold_100'] - pivoted['gold_200']

pivoted_with_team = pivoted.merge(member_teams, on='matchId', how='inner')
pivoted_with_team['memberGoldLead'] = np.where(
    pivoted_with_team['teamId'] == 100,
    pivoted_with_team['goldDiff'],
    -pivoted_with_team['goldDiff']
)

pivoted_with_win = pivoted_with_team.merge(
    df_matches[['matchId', 'team100Win']], on='matchId', how='left'
)
pivoted_with_win['win'] = (
    (pivoted_with_win['teamId'] == 100) & pivoted_with_win['team100Win']
) | (
    (pivoted_with_win['teamId'] == 200) & ~pivoted_with_win['team100Win']
)

avg_lead = pivoted_with_win.groupby(['timestampMin', 'win'])['memberGoldLead'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for win_val, label, color in [(True, '勝ち試合', '#2ecc71'), (False, '負け試合', '#e74c3c')]:
    data = avg_lead[avg_lead['win'] == win_val]
    ax.plot(data['timestampMin'], data['memberGoldLead'], label=label, color=color, linewidth=2)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(
    avg_lead[avg_lead['win'] == True]['timestampMin'],
    avg_lead[avg_lead['win'] == True]['memberGoldLead'],
    alpha=0.1, color='#2ecc71'
)
ax.fill_between(
    avg_lead[avg_lead['win'] == False]['timestampMin'],
    avg_lead[avg_lead['win'] == False]['memberGoldLead'],
    alpha=0.1, color='#e74c3c'
)
ax.set_xlabel('試合時間（分）')
ax.set_ylabel('平均ゴールドリード（味方チーム視点）')
ax.set_title('勝ち/負け試合の平均ゴールドリード推移')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 逆転勝利 / 逆転負けパターン分析
at_15 = pivoted_with_win[pivoted_with_win['timestampMin'] == 15.0].copy()

if not at_15.empty:
    at_15['leadAt15'] = at_15['memberGoldLead'] > 0

    reversal = at_15.groupby(['leadAt15', 'win']).size().unstack(fill_value=0)
    reversal.index = ['15分不利', '15分有利']
    reversal.columns = ['敗北', '勝利']

    print('15分時点の有利/不利 × 最終結果:')
    print(reversal)
    print()

    total_behind = reversal.loc['15分不利'].sum()
    comeback_wins = reversal.loc['15分不利', '勝利'] if '勝利' in reversal.columns else 0
    total_ahead = reversal.loc['15分有利'].sum()
    thrown = reversal.loc['15分有利', '敗北'] if '敗北' in reversal.columns else 0

    print(f'逆転勝利率（15分不利→勝利）: {comeback_wins/total_behind*100:.1f}% ({comeback_wins}/{total_behind})')
    print(f'スノーボール率（15分有利→勝利）: {(total_ahead-thrown)/total_ahead*100:.1f}% ({total_ahead-thrown}/{total_ahead})')
    print(f'逆転負け率（15分有利→敗北）: {thrown/total_ahead*100:.1f}% ({thrown}/{total_ahead})')
else:
    print('15分データなし')

In [ ]:
# メンバー別 得意な試合時間帯
df_p_m = df_players[df_players['riotId'].isin(MEMBERS)].copy()
df_p_m = df_p_m.merge(
    df_matches[['matchId', 'gameDurationMin']], on='matchId', how='left'
)
df_p_m['durBin'] = pd.cut(
    df_p_m['gameDurationMin'], bins=dur_bins, labels=dur_labels
)

member_dur_wr = df_p_m.groupby(['summonerName', 'durBin'], observed=False).agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
).reset_index()
member_dur_wr['winrate_pct'] = (member_dur_wr['winrate'] * 100).round(1)

pivot_dur = member_dur_wr.pivot_table(
    index='summonerName', columns='durBin', values='winrate_pct'
)

if not pivot_dur.empty:
    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot_dur) * 0.6)))
    sns.heatmap(pivot_dur, annot=True, fmt='.0f', cmap='RdYlGn', center=50, ax=ax)
    ax.set_title('メンバー × 試合時間帯 勝率ヒートマップ (%)')
    ax.set_xlabel('試合時間')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()